# Virágfelismerés – Végső bemutató és összegzés

Ebben a notebookban bemutatom a korábban betanított virágfelismerő modellt, kipróbálom egy új képen, és összegzem az eredményeket.

## CRISP-DM módszertan alkalmazása

A projekt során a CRISP-DM (Cross Industry Standard Process for Data Mining) módszertant követtem.

### 1. Üzleti megértés
A cél egy olyan modell létrehozása volt, amely képek alapján képes különböző virágfajtákat felismerni.

### 2. Adatmegértés
Megvizsgáltam a rendelkezésre álló képadatbázist, amely több virágfaj képét tartalmazza külön mappákban.

### 3. Adatelőkészítés
A képeket egységes méretre alakítottam, majd tanító és validációs halmazokra bontottam.

### 4. Modellezés
Konvolúciós neurális hálózatot (CNN) hoztam létre a képosztályozási feladat megoldására, és több modellt is összehasonlítottam.

### 5. Kiértékelés
A modell teljesítményét pontosság, veszteség, konfúziós mátrix és vizualizációk segítségével értékeltem.

### 6. Alkalmazás
A végső modell új, felhasználó által feltöltött képeken is képes predikciót végezni.

## A notebook célja

Ebben a notebookban:
- betöltöm a korábban elmentett modellt,
- feltöltök egy új virágképet,
- a modellen predikciót futtatok,
- magyar nyelven jelenítem meg az eredményt,
- vizualizálom a predikció valószínűségeit,
- összegzem a projekt eredményeit és fejlesztési lehetőségeit.

In [ ]:
import os

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import io

from pathlib import Path
from IPython.display import display
from PIL import Image

# Projektgyökér meghatározása helyi Jupyter és Colab futtatáshoz.
cwd = Path.cwd()
possible_roots = [
    cwd,
    cwd.parent,
    Path("/content/adatelemzesi-projekt"),
    Path("/content/drive/MyDrive/adatelemzesi-projekt"),
]

PROJECT_ROOT = None
for candidate in possible_roots:
    if (candidate / "notebooks").exists() and (candidate / "README.md").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    if cwd.name == "notebooks" and (cwd.parent / "README.md").exists():
        PROJECT_ROOT = cwd.parent
    else:
        PROJECT_ROOT = cwd

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if NOTEBOOKS_DIR.exists() and Path.cwd() != NOTEBOOKS_DIR:
    os.chdir(NOTEBOOKS_DIR)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"

for directory in [FIGURES_DIR, TABLES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "flower_classifier.keras"
print(f"Projekt gyökérmappa: {PROJECT_ROOT}")
print(f"Várt modellútvonal: {MODEL_PATH}")


## A modell betöltése

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"A modellfájl nem található: {MODEL_PATH}. "
        "Először futtasd a notebooks/03_modeling.ipynb notebookot (vagy helyezz el egy előre betanított modellt ezen az útvonalon)."
    )

model = tf.keras.models.load_model(MODEL_PATH)
print("Modell betöltve.")


## Kategóriák magyar megnevezésekkel

In [ ]:
class_names = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']

hungarian_labels = {
    "daisy": "százszorszép",
    "dandelion": "pitypang",
    "roses": "rózsa",
    "sunflowers": "napraforgó",
    "tulips": "tulipán"
}

## Kép feltöltése

Az alábbi fájlfeltöltő mező segítségével kiválasztható egy új virágkép, amelyen a modell predikciót végez.

In [ ]:
upload = widgets.FileUpload(
    accept='image/*',
    multiple=False
)

display(upload)
print("Válassz ki egy virág képet a feltöltéshez.")

## A feltöltött kép feldolgozása

In [ ]:
if len(upload.value) == 0:
    print("Először tölts fel egy képet!")
else:
    if isinstance(upload.value, tuple):
        file_info = upload.value[0]
    else:
        file_info = list(upload.value.values())[0]

    content = file_info["content"]

    img = Image.open(io.BytesIO(content)).convert("RGB")
    img = img.resize((180, 180))

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Feltöltött kép")
    uploaded_preview_path = FIGURES_DIR / "04_uploaded_image_preview.png"
    plt.savefig(uploaded_preview_path, dpi=300, bbox_inches="tight")
    print(f"Ábra mentve: {uploaded_preview_path}")
    plt.show()

    img_array = np.array(img)
    img_array = np.expand_dims(img_array, 0)

    print("Kép betöltve.")


## Predikció

In [ ]:
if 'img_array' not in locals():
    print("Először tölts fel és dolgozz fel egy képet!")
else:
    predictions = model.predict(img_array)

    score = tf.nn.softmax(predictions[0])

    predicted_class = class_names[np.argmax(score)]
    predicted_class_hu = hungarian_labels[predicted_class]

    confidence = 100 * np.max(score)

    print(f"A modell szerint ez a virág: {predicted_class_hu}")
    print(f"A predikció megbízhatósága: {confidence:.2f}%")

    prediction_probabilities_df = pd.DataFrame({
        "kategoria_angol": class_names,
        "kategoria_magyar": [hungarian_labels[name] for name in class_names],
        "valoszinuseg": score.numpy()
    })
    prediction_probabilities_path = TABLES_DIR / "04_prediction_probabilities.csv"
    prediction_probabilities_df.to_csv(prediction_probabilities_path, index=False)
    print(f"Táblázat mentve: {prediction_probabilities_path}")

    prediction_summary_df = pd.DataFrame([{
        "prediktalt_kategoria_angol": predicted_class,
        "prediktalt_kategoria_magyar": predicted_class_hu,
        "biztonsag_szazalek": float(confidence)
    }])
    prediction_summary_path = TABLES_DIR / "04_prediction_summary.csv"
    prediction_summary_df.to_csv(prediction_summary_path, index=False)
    print(f"Táblázat mentve: {prediction_summary_path}")


## Predikció valószínűségeinek vizualizációja

In [ ]:
if 'score' not in locals():
    print("Először futtasd le a predikciót!")
else:
    probabilities = score.numpy()

    plt.figure(figsize=(8, 4))
    plt.bar(
        [hungarian_labels[name] for name in class_names],
        probabilities
    )

    plt.title("Predikció valószínűségek")
    plt.xlabel("Virág kategória")
    plt.ylabel("Valószínűség")
    plt.xticks(rotation=15)
    prediction_plot_path = FIGURES_DIR / "04_prediction_probabilities.png"
    plt.savefig(prediction_plot_path, dpi=300, bbox_inches="tight")
    print(f"Ábra mentve: {prediction_plot_path}")
    plt.show()


## A modell pontosságának értékelése

A projekt során több modellt és paraméterezést vizsgáltam. Az összehasonlítás alapján a legjobb modell stabilabb validációs pontosságot mutatott, ezért ezt mentettem el végső modellként.

A validációs pontosság azt mutatja, hogy a modell nemcsak a tanító adatokon, hanem új, korábban nem látott képeken is képes megfelelő teljesítményt nyújtani.

A legnehezebben felismerhető virágfajok általában azok, amelyek vizuálisan hasonlítanak egymásra, például a százszorszép és a pitypang, illetve bizonyos esetekben a rózsa és a tulipán.

## Következtetés

A létrehozott konvolúciós neurális hálózat alkalmas arra, hogy különböző virágfajtákat képek alapján osztályozzon. A modell képes új képeken is predikciót végezni, és a kapott eredmények alapján a megoldás működőképesnek tekinthető.

A projekt során sikerült megvalósítani az adatbetöltést, az adatelőkészítést, a modell tanítását, a teljesítmény értékelését és a predikció gyakorlati bemutatását.

## Fejlesztési lehetőségek

A modell teljesítménye több módon javítható:

1. Nagyobb és változatosabb dataset használata
2. Adatbővítés alkalmazása, például forgatás, tükrözés és nagyítás
3. Mélyebb neurális háló architektúra használata
4. Transzfer tanulás alkalmazása előre tanított modellekkel, például MobileNet vagy ResNet
5. Több epoch használata a tanítás során

Ezek a módszerek javíthatják a modell általánosítási képességét és növelhetik a pontosságot.